# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mohidraheel/Machine-Learning-Practice/blob/main/work/notebooks/w05_model.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
!pip -q install duckdb huggingface_hub pandas numpy scikit-learn

In [2]:
import os
import json
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd

from google.colab import userdata
from huggingface_hub import login
from IPython.display import display

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 180)

print("Libraries loaded.")

Libraries loaded.


In [3]:
hf_token = userdata.get("HF_TOKEN")

if not hf_token:
    raise ValueError(
        "HF_TOKEN was not found. Add it in Colab Secrets and enable notebook access."
    )

os.environ["HF_TOKEN"] = hf_token
login(token=hf_token, add_to_git_credential=False)

con = duckdb.connect()
con.execute("INSTALL httpfs")
con.execute("LOAD httpfs")
con.execute("SET secret_directory='/tmp'")

con.execute(f'''
    CREATE OR REPLACE SECRET hf_secret (
        TYPE HUGGINGFACE,
        TOKEN '{hf_token}'
    )
''')

WAREHOUSE_PATH = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/**/*.parquet"
)

con.execute(f'''
    CREATE OR REPLACE VIEW warehouse AS
    SELECT *
    FROM read_parquet(
        '{WAREHOUSE_PATH}',
        hive_partitioning=true
    )
''')

print("Warehouse connected.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Warehouse connected.


In [4]:
schema_df = con.execute("DESCRIBE warehouse").df()
display(schema_df)

required_columns = [
    "report_date",
    "client_hash_id",
    "content_hash_id",
    "gsc_data_available",
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_data_available",
    "ga4_sessions",
]

available_columns = schema_df["column_name"].tolist()

missing_columns = [
    column
    for column in required_columns
    if column not in available_columns
]

if missing_columns:
    raise KeyError(f"Missing required warehouse columns: {missing_columns}")

print("Required columns are available.")

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


Required columns are available.


In [ ]:
modeling_sql = '''
WITH march AS (
    SELECT
        client_hash_id AS client_id,
        content_hash_id AS content_id,

        SUM(COALESCE(gsc_impressions, 0)) AS impressions_31d,
        SUM(COALESCE(gsc_clicks, 0)) AS clicks_31d,

        CASE
            WHEN SUM(COALESCE(gsc_impressions, 0)) > 0
            THEN SUM(COALESCE(gsc_clicks, 0)) * 1.0
                 / SUM(COALESCE(gsc_impressions, 0))
            ELSE NULL
        END AS ctr_31d,

        CASE
            WHEN SUM(
                CASE
                    WHEN gsc_avg_position IS NOT NULL
                    THEN COALESCE(gsc_impressions, 0)
                    ELSE 0
                END
            ) > 0
            THEN SUM(
                CASE
                    WHEN gsc_avg_position IS NOT NULL
                    THEN gsc_avg_position * COALESCE(gsc_impressions, 0)
                    ELSE 0
                END
            ) * 1.0
            / SUM(
                CASE
                    WHEN gsc_avg_position IS NOT NULL
                    THEN COALESCE(gsc_impressions, 0)
                    ELSE 0
                END
            )
            ELSE NULL
        END AS weighted_position_31d,

        SUM(COALESCE(ga4_sessions, 0)) AS sessions_31d,
        MAX(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS has_ga4_data,
        COUNT(*) AS march_source_rows

    FROM warehouse

    WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
      AND gsc_data_available IS TRUE

    GROUP BY
        client_hash_id,
        content_hash_id

    HAVING SUM(COALESCE(gsc_impressions, 0)) >= 100
),

april AS (
    SELECT
        client_hash_id AS client_id,
        content_hash_id AS content_id,
        SUM(COALESCE(gsc_clicks, 0)) AS april_clicks,
        COUNT(*) AS april_source_rows

    FROM warehouse

    WHERE report_date BETWEEN DATE '2026-04-01' AND DATE '2026-04-30'
      AND gsc_data_available IS TRUE

    GROUP BY
        client_hash_id,
        content_hash_id
)

SELECT
    m.*,
    a.april_clicks,
    a.april_source_rows,

    CASE
        WHEN m.clicks_31d > 0
        THEN (a.april_clicks - m.clicks_31d) * 1.0 / m.clicks_31d
        ELSE NULL
    END AS future_click_change_pct,

    CASE
        WHEN m.clicks_31d > 0
         AND a.april_clicks <= m.clicks_31d * 0.90
        THEN 1
        ELSE 0
    END AS click_decline_label

FROM march AS m

INNER JOIN april AS a
    USING (client_id, content_id)
'''

model_frame = con.execute(modeling_sql).df()

print("Modeling rows:", len(model_frame))
display(model_frame.head(10))

In [ ]:
assert len(model_frame) > 0, "The modeling frame is empty."
assert model_frame[["client_id", "content_id"]].notna().all().all()
assert model_frame["impressions_31d"].ge(100).all()
assert model_frame["click_decline_label"].isin([0, 1]).all()

print("Positive-label rate:", round(model_frame["click_decline_label"].mean(), 4))
print("Unique clients:", model_frame["client_id"].nunique())

display(
    model_frame[
        [
            "impressions_31d",
            "clicks_31d",
            "ctr_31d",
            "weighted_position_31d",
            "sessions_31d",
            "april_clicks",
            "future_click_change_pct",
            "click_decline_label",
        ]
    ].describe()
)

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

March metrics are used as features. April clicks are used only to build the future outcome.

Eligibility requires:

- March GSC data
- At least 100 March impressions
- At least one April GSC row

The label is `1` when April clicks are at least 10% below March clicks. Pages with zero March clicks remain eligible; for them, the label is `0` because a percentage decline cannot be established from a zero baseline.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.